In [4]:
import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup
import time
from io import StringIO

In [5]:
url = 'https://arp.sn/bases-de-donnees/medicaments-autorises'
reponse = requests.get(url)
soup = BeautifulSoup(reponse.text, 'html.parser')


In [6]:
!pip install lxml

   ---------------------------------------- 0.0/4.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/4.1 MB ? eta -:--:--
   ----- ---------------------------------- 0.5/4.1 MB 4.2 MB/s eta 0:00:01
   ------------------ --------------------- 1.8/4.1 MB 4.8 MB/s eta 0:00:01
   ------------------------- -------------- 2.6/4.1 MB 4.6 MB/s eta 0:00:01
   --------------------------------- ------ 3.4/4.1 MB 4.2 MB/s eta 0:00:01
   ---------------------------------------- 4.1/4.1 MB 4.2 MB/s  0:00:01



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: C:\Users\Surface\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [7]:
base_url = "https://arp.sn/bases-de-donnees/medicaments-autorises"
all_data = []
total_pages = 299

# En-têtes réels identifiés dans la structure
columns_names = [
    "Nom du Médicament", 
    "DCI", 
    "Dosage", 
    "Présentation", 
    "Forme Galénique", 
    "Voie d'administration", 
    "Laboratoire", 
    "Classe Thérapeutique"
]

print("Début de l'extraction de toutes les pages d'AMM...")

for page in range(1, total_pages + 1):
    url = f"{base_url}?page={page}"
    try:
        response = requests.get(url, timeout=15)
        if response.status_code == 200:
            # Lecture des tables de la page
            html_io = StringIO(response.text)
            tables = pd.read_html(html_io)
            
            # On filtre pour trouver la table contenant les médicaments
            # La table cible a généralement 8 colonnes
            page_added = False
            for df in tables:
                if df.shape[1] == 8 and not df.empty:
                    # Si la première ligne contient les noms de colonnes ou des valeurs vides, on les nettoie si besoin
                    df.columns = columns_names
                    all_data.append(df)
                    page_added = True
            
            if page_added:
                print(f"Page {page}/{total_pages} récupérée avec succès.", end='\r')
            else:
                print(f"Page {page}/{total_pages} : Aucune table de données valide trouvée.")
        else:
            print(f"Page {page}/{total_pages} : Erreur HTTP {response.status_code}")
    except Exception as e:
        print(f"Page {page}/{total_pages} : Erreur détectée -> {str(e)}")
    
    # Un court délai pour ne pas surcharger le serveur
    time.sleep(0.5)

# Fusion et création du fichier Excel final
if all_data:
    final_df = pd.concat(all_data, ignore_index=True)
    # Supprimer les éventuelles lignes doublons d'en-tête ou entièrement vides
    final_df = final_df.dropna(how='all')
    
    output_file = 'medicaments_amm_senegal.csv'
    final_df.to_csv(output_file, index=False)
    print(f"\nExtraction terminée ! {len(final_df)} médicaments extraits et enregistrés dans '{output_file}'.")
    display(final_df.head())
else:
    print("\nAucune donnée n'a pu être extraite.")

Début de l'extraction de toutes les pages d'AMM...
Page 299/299 récupérée avec succès.
Extraction terminée ! 7425 médicaments extraits et enregistrés dans 'medicaments_amm_senegal.csv'.


,Nom du Médicament,DCI,Dosage,Présentation,Forme Galénique,Voie d'administration,Laboratoire,Classe Thérapeutique
0,5 DF,CHLORURE DE SODIUM,5 g,FL/100ml,solution pour perfusion,intraveineuse,ACULIFE HEALTHCARE PVT,SOLUTION POUR REHYDRATATION
1,5-FLUOROURACILE SANDOZ,5-FLUOROURACILE,250 mg / 5 ml,FL/5ml,solution injectable,injectable,SANDOZ,ANTICANCEREUX
2,5-FLUOROURACILE SANDOZ,5-FLUOROURACILE,1000 mg / 20 ml,FL/20ml,solution injectable,intraveineuse,SANDOZ,ANTICANCEREUX
3,A-CERUMEN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ABACAVIR,ABACAVIR,300 mg,B/60,comprimé,orale,HETERO,ANTIRETROVIRAL
